[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-0/basics.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/56295530-getting-set-up-video-guide)

# LangChain Academy

Welcome to LangChain Academy! 

## Context

At LangChain, we aim to make it easy to build LLM applications. One type of LLM application you can build is an agent. There’s a lot of excitement around building agents because they can automate a wide range of tasks that were previously impossible. 

In practice though, it is incredibly difficult to build systems that reliably execute on these tasks. As we’ve worked with our users to put agents into production, we’ve learned that more control is often necessary. You might need an agent to always call a specific tool first or use different prompts based on its state. 

To tackle this problem, we’ve built [LangGraph](https://docs.langchain.com/oss/python/langgraph/overview) — a framework for building agent and multi-agent applications. Separate from the LangChain package, LangGraph’s core design philosophy is to help developers add better precision and control into agent workflows, suitable for the complexity of real-world systems.

## Course Structure

The course is structured as a set of modules, with each module focused on a particular theme related to LangGraph. You will see a folder for each module, which contains a series of notebooks. A video will accompany each notebook to help walk through the concepts, but the notebooks are also stand-alone, meaning that they contain explanations and can be viewed independently of the videos. Each module folder also contains a `studio` folder, which contains a set of graphs that can be loaded into [LangSmith Studio](https://docs.langchain.com/langsmith/quick-start-studio), our IDE for building LangGraph applications.

## Setup

Before you begin, please follow the instructions in the `README` to create an environment and install dependencies.

## Chat models

In this course, we'll use Chat Models, which take a sequence of messages as input and return messages as output. LangChain supports many models via [third-party integrations](https://docs.langchain.com/oss/python/integrations/chat). By default, the course will use  [ChatOpenAI](https://docs.langchain.com/oss/python/integrations/chat/openai) because it is both popular and performant. As noted, please ensure that you have an `OPENAI_API_KEY`.

Let's check that your `OPENAI_API_KEY` is set and, if not, you will be asked to enter it.

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langchain_openai langchain_core langchain_community langchain-tavily

In [ ]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

# _set_env("OPENAI_API_KEY")
_set_env("GOOGLE_API_KEY")

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

[Here](https://docs.langchain.com/oss/python/langchain/models) is a useful how-to for all the things that you can do with chat models, but we'll show a few highlights below. If you've run `pip install -r requirements.txt` as noted in the README, then you've installed the `langchain-openai` package. With this, we can instantiate our `ChatOpenAI` model object. You can see pricing for various models [here](https://openai.com/api/pricing/). The notebooks will default to `gpt-4o` because it offers a good balance of quality, price, and speed, but you can also opt for the lower-priced `gpt-3.5` series or more recent models.

There are [a few standard parameters](https://docs.langchain.com/oss/python/langchain/models#parameters) that we can set with chat models. Two of the most common are:

* `model`: the name of the model
* `temperature`: the sampling temperature

`Temperature` controls the randomness or creativity of the model's output where low temperature (close to 0) is more deterministic and focused outputs. This is good for tasks requiring accuracy or factual responses. High temperature (close to 1) is good for creative tasks or generating varied responses. 

In [2]:
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
# gpt4o_chat = ChatOpenAI(model="gpt-4o", temperature=0)
# gpt35_chat = ChatOpenAI(model="gpt-3.5-turbo-0125", temperature=0)

gemini35_chat = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0)

Chat models in LangChain have a number of [default methods](https://reference.langchain.com/python/langchain_core/runnables). For the most part, we'll be using:

* [stream](https://docs.langchain.com/oss/python/langchain/models#stream): stream back chunks of the response
* [invoke](https://docs.langchain.com/oss/python/langchain/models#invoke): call the chain on an input

And, as mentioned, chat models take [messages](https://docs.langchain.com/oss/python/langchain/messages) as input. Messages have a role (that describes who is saying the message) and a content property. We'll be talking a lot more about this later, but here let's just show the basics.

In [3]:
from langchain_core.messages import HumanMessage

# Create a message
msg = HumanMessage(content="Hello world", name="Lance")

# Message list
messages = [msg]

# Invoke the model with a list of messages 
# gpt4o_chat.invoke(messages)
gemini35_chat.invoke(messages)

AIMessage(content=[{'type': 'text', 'text': 'Hello, World! 👋 How can I help you today?', 'extras': {'signature': 'EsAHCr0HAQw51sdbEBzk8MvW9MMLxP0vDUMKJYDejiYCF8JM68NDlwK0i82nUudjmpX8G113HQlEb4Ap3f1guCwct97uFvBIV6PMbsTSgsIUUGR+GoGh+Cy4KoXKwT86Qm9bCgMci1yQgoFWPaalmG4DNyXikbyLBJjdZE+Xw7zVmt07C9VuZBX302dlOsfxxMK3U/yGHEjBgJDn8VVHjLQfda2qwhoumY6aQXeWh0Jj7tKkCe51BJf0OaHyBiC+FTSkNtPA2LRvdXQZff/iVMEJLk8T2AYNIW04PkmG8/CPsAYt9UypAunoEoj5wUt6f8UC1uDTqGzyYuEFYi2hhga4aJT+UZEEenb8hqz6kzomeTGiMZy0nbOXcTBqDdLnvq3WgDPFRROVDd9jdgDsg6Ujw4pYAOZ8ZJGa3S1JT3xWD//Vcn3dOr6N54r+dniqOCbjAoCJvF+qYj6133m6nfMELV3Nl9RSXLsGP7LHmjQtcOQ01KpayURTCrQzp9HbPsyfBC/h9u/bm+CMfEnm2Ywy2dtr3hYGhd/Ioe+4tcELoMnKxhHjVDkf6epVbqSsZWPOGskDE5SlfnfIZtckZKUdZbeTVRx2ecP4wcCmlFtm2pc0nSjsoQHaigXfwxoMDfL37pZdFJ4KDjiKAZoSwfn4nQJ3jHZqaYS5MPPfQpdXMl/2hXuuf9t11vY66B9TSiBCmn9MZokWJDFlMu7Q3PODKETL9DjUBwtxRnL++d1nGR6BH1vi0VaGvMaxqOYP/VLBN2zpGG+5UOsH+dfqr2k8NK53Rlvh6p9jBqZH74Rxal9D1XARsjBagNnEOnZML3EuA33W/ffWHHyMdn9HXPFVt3GgbcGDzSRmp8YR1tJrAXRZgUD/3V

We get an `AIMessage` response. Also, note that we can just invoke a chat model with a string. When a string is passed in as input, it is converted to a `HumanMessage` and then passed to the underlying model.


In [4]:
# gpt4o_chat.invoke("hello world")
gemini35_chat.invoke("hello world")

AIMessage(content=[{'type': 'text', 'text': 'Hello, World! 👋 \n\nHow can I help you today?', 'extras': {'signature': 'Eu4GCusGAQw51secgqpwbHeShyasf6dhFsBMA6r8JRfK6rV2KgZiHeCE4AKDOTsBWhCHeVWAiEbP31aMC/EBkMT4bnhLK8W4q4NEiwL4UjgK2zObHwhIl99rGjQwWZTd0gRMOxVmHv0M6PjCQgcllAJB/6t7Ko6IvD6TSp3PVg8Y6PaQOFhk99wdYStS4kOtRV2aS48R7v322wfsM1hNII9TUgHwkhPBlQvBG5gvkYmWGECUT2YptaauBAfH9ryrALH8mn0vIPZ7wvJGDbGzyq0eKWLo6rfnc3kGA1pAFVcsG4JC197yiGuCAo2iBSP54aIh1oE5GxNYhVK+y8IdxrM1Ovupnv/wVZ7PB0aiyLbZa1ZOZNszJEzB7BVdW43DF4Fc5RPiQPyGpw6G5uAshRegn2D3qdmH7HidExltPrD1/ZBjy8QTlbVa2EchH7WMuNZE0DEqjnDrGIcXpQpYqnugifk6D+vwKWOXdXivt53y/eSTBVwX54EhsdhEfMMkO0EUaWAUvIYgxPly1+DEwzecfRQF6RRN7Ct1WKiFlWMYmF1bnQYPPOzOyrfp7npXro3JcTFOarQEvkTuh+4t/9EbEwXRl9I4gmC/sHpMr60ZVBWbc8lUuDYFsAitS+OhziW/EGygwZifBQyXMzGjDeyYYjDJ6ZMnk17KsuEQjzuGYMM6433WyXosPKlYKrCKnHeLVO8i3e+CK6uapz4PCypDdNepvXO7boYChV0PAIrL/EwJS+EvqT/eEudUcaOUO9/k16XVUZQtxbZ+XnZ369gBMBfhHsZTMgxn+k306Fua+SsOWeuKh8ZAj9CqO0BiRP4c16dRsw2Gh7gKfIR1BMRZMCPZh38wDmk9ZDsNaLSLbdKEK7

In [5]:
# gpt35_chat.invoke("hello world")
gemini35_chat.invoke("hello world")

AIMessage(content=[{'type': 'text', 'text': 'Hello, World! 👋 \n\nHow can I help you today?', 'extras': {'signature': 'Eu4GCusGAQw51seqrkSON6bBaLLzwY27xErJwaxrtQEnQOy9hOJTnRaRGfMljsgBKW/o2p/2fURbFK5AW+9kDgK6N8n9mTGyEYIMXANWrSQzEb3z7Vrhr0fISbCx/JLLXzZ38ZsGGEysaboJCwy+Yffbr/CcSd2OFykcKNgLzTIWLDUbdrw4gndlNkK+z3MJvFduxT9RlxI62QjW1GfkhrhXe7ng/8ymUa8ek+fp+v0HHsfE9XuZpprdcEwgt6DAH8oAfwlz/wfXosda1l79tewn5MXHHoZi6RqOOD4Gmt31xeIOJBMAfVRrUQ0DtHrcj/+sQ2YzcM0MGl6Nhm2MU/na8FPuLHxE1vWzuhti7Iz0IxBz+rWmvcW+RvFaBieonxibIRnerTHUjKFkf1w2i2fAuZBLMahOtQXudagQObWRNmxNn1jCyRMqGb7ty9BypMB0ztPWpybzOzhkRyaEBXKs8i8Hqdv/pukxPQFLzzU2AlsFig4tiLMFQJt2Cm7sJ6ZaVR0i5kIZGIsvYH/XAunoqCVdigg3wQFj8PyUFGCFk1FPjxypeY5ivonPx+NyLzSdka82pZ51t51EDBoT2HYNsU5wDMhnVaY8HknL+fE0jz44DiKHo2ycitrqLVcFPnLunutPPBs07k5zbLG3a4TXMNENru8Nm1HaPlTTCfIwNQ5SuWkwPgG0thBrLRyqmfQCdJtG+WRbfQJJO+Qj2bd2OhyGF9P9NIE8NhD90qtcDFDElg3QCnuhv8uIb7+a7dv6ymUz0qxbVNfXUC6+nxFyTEX1o/6xL9034rykkr5bVL4fQ1/yxyXhHGqXaAJqKlFIc6v8aSkiGpW0Z9C0/Sf9Tc7/tTQQlPsq62B5mP6fWYmE+V

The interface is consistent across all chat models and models are typically initialized once at the start up each notebooks. 

So, you can easily switch between models without changing the downstream code if you have strong preference for another provider.


## Search Tools

You'll also see [Tavily](https://tavily.com/) in the README, which is a search engine optimized for LLMs and RAG, aimed at efficient, quick, and persistent search results. As mentioned, it's easy to sign up and offers a generous free tier. Some lessons (in Module 4) will use Tavily by default but, of course, other search tools can be used if you want to modify the code for yourself.

In [ ]:
# _set_env("TAVILY_API_KEY")

In [6]:
from langchain_tavily import TavilySearch  # updated at 1.0

tavily_search = TavilySearch(max_results=3)

data = tavily_search.invoke({"query": "What is LangGraph?"})
search_docs = data.get("results", data)

In [7]:
search_docs

[{'url': 'https://www.ibm.com/think/topics/langgraph',
  'title': 'What is LangGraph? - IBM',
  'content': 'LangGraph, created by LangChain, is an open source AI agent framework designed to build, deploy and manage complex generative AI agent workflows. It provides a set of tools and libraries that enable users to create, run and optimize large language models (LLMs) in a scalable and efficient manner. At its core, LangGraph uses the power of graph-based architectures to model and manage the intricate relationships between various components of an AI agent workflow. The following example can offer a clearer understanding of LangGraph: Think about these graph-based architectures as a powerful configurable map, a “Super-Map.” Users can envision the AI workflow as being “The Navigator” of this “Super-Map.” Finally, in this example, the user is “The Cartographer.” In this sense, the navigator charts out the optimal routes between points on the “Super-Map,” all of which are created by “The 